# Time Series Data Preparation

### 1.1 Loading Already Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pickle
import gc
import warnings
warnings.filterwarnings('ignore')

# Load data after EDA (already cleaned, without duplicates)
df = pd.read_csv('data/clean_primary_dedup.csv', parse_dates=['время измерения'])
print(f"   Measurements loaded: {len(df):,}")
print(f"   Unique patients: {df['id пациента'].nunique():,}")

# Add date without time for aggregation
df['дата'] = df['время измерения'].dt.date
df['дата'] = pd.to_datetime(df['дата'])

# Check for pulse pressure (create if not exists)
if 'пульсовое_давление' not in df.columns:
    df['пульсовое_давление'] = df['САД'] - df['ДАД']

# Load group information from patient_program
patient_program = pd.read_csv('data/patient_program.csv')
df = df.merge(
    patient_program[['id пациента', 'группа наблюдения']].drop_duplicates('id пациента'),
    on='id пациента',
    how='left'
)

### 1.2 Creating Daily Aggregates for Time Series

In [ ]:
# Daily aggregates - only needed fields for DTW
daily_stats = df.groupby(['id пациента', 'дата']).agg({
    'САД': 'mean',
    'ДАД': 'mean',
    'ЧП': 'mean',
    'пульсовое_давление': 'mean'
}).round(1).reset_index()
print(f"   Daily records created: {len(daily_stats):,}")
print(f"   Unique patients: {daily_stats['id пациента'].nunique():,}")

"Daily records created: 913,553"
This is the number of days with measurements across all patients:

We have 10,003 patients

In total they have 913,553 days when measurements were taken

On average: 913,553 / 10,003 ≈ 91 days of measurements per patient

But importantly: this is not 91 consecutive days, but the total number of days when the patient took measurements. For example, a patient could measure 10 days in January, 15 in February, 5 in March - total 30 days.

### 1.3 Filtering Patients for DTW (using criteria from the plan)

In [ ]:
# Load CSE for analysis
kzs = pd.read_csv('data/kzs.csv', parse_dates=['дата, время формирования КЗС'])
kzs['id пациента'] = kzs['id пациента'].astype(str)
print(f"CSE loaded: {len(kzs)}")

# Get list of patients with CSE
patients_with_kzs = set(kzs['id пациента'].unique())
print(f"Total patients with CSE: {len(patients_with_kzs)}")
print("\nAnalyzing observation duration for patients with CSE.")

# Get statistics for ALL patients from daily_stats (ONCE)
all_patients_stats = daily_stats.groupby('id пациента').agg(
    days=('дата', 'count'),
    measurements=('САД', 'count')
).reset_index()
all_patients_stats['id пациента'] = all_patients_stats['id пациента'].astype(str)
print(f"  Statistics for all patients: {len(all_patients_stats)}")

# Filter by patients with CSE
kzs_patients_stats = all_patients_stats[all_patients_stats['id пациента'].isin(patients_with_kzs)].copy()
kzs_patients_stats = kzs_patients_stats.rename(columns={'id пациента': 'patient'})
print(f"  Patients with CSE found in daily_stats: {len(kzs_patients_stats)}")
print(f"\nStatistics for patients with CSE:")
print(f"  Median observation days: {kzs_patients_stats['days'].median():.0f}")
print(f"  Patients with ≥30 days: {(kzs_patients_stats['days'] >= 30).sum()}")
print(f"  Patients with 14-29 days: {((kzs_patients_stats['days'] >= 14) & (kzs_patients_stats['days'] < 30)).sum()}")

# SELECTION CRITERIA
min_days_clustering = 30
min_days_classification = 14
max_patients_clustering = 7389
max_patients_classification = 4000

# 1. Patients for CLUSTERING
clustering_candidates = kzs_patients_stats[kzs_patients_stats['days'] >= min_days_clustering]['patient'].tolist()
print(f"\nCandidates for clustering: {len(clustering_candidates)}")

# Sort by number of days
clustering_candidates = sorted(clustering_candidates, 
                              key=lambda x: kzs_patients_stats[kzs_patients_stats['patient']==x]['days'].values[0],
                              reverse=True)

clustering_patients = clustering_candidates[:max_patients_clustering]
print(f"Selected for clustering: {len(clustering_patients)} patients")

# 2. Patients for CLASSIFICATION
classification_candidates = kzs_patients_stats[kzs_patients_stats['days'] >= min_days_classification]['patient'].tolist()
classification_patients = classification_candidates[:max_patients_classification]
print(f"Selected for classification: {len(classification_patients)} patients")

# Dictionary of CSE count per patient
kzs_count_per_patient = kzs.groupby('id пациента').size().to_dict()

Is a minimum of 30 days of observation mandatory?

    Too short series (<30 days) give unreliable patterns. Trends and cyclicity cannot be observed. Random fluctuations outweigh real patterns.

### 1.4 Creating Time Series Dictionaries for GT-DTW

In [ ]:
# Convert IDs to string once for the entire dataframe
daily_stats['id_patient_str'] = daily_stats['id пациента'].astype(str)
all_needed_patients = set(clustering_patients) | set(classification_patients)
print(f"Total unique patients for processing: {len(all_needed_patients)}")

# Group data by patient (one operation instead of 7389!)
grouped = daily_stats.groupby('id_patient_str')
patient_groups = {}

# Create dictionaries via groupby (lightning fast!)
patient_dates = {}
patient_series = {'САД': {}, 'ДАД': {}, 'ЧП': {}, 'пульсовое_давление': {}}

# Get all data in one pass
for patient, group in grouped:
    if patient not in all_needed_patients:
        continue
        
    # Sort once for each patient
    group_sorted = group.sort_values('дата')
    
    # Save dates
    patient_dates[patient] = group_sorted['дата'].values
    
    # Save series for all metrics
    for metric in ['САД', 'ДАД', 'ЧП', 'пульсовое_давление']:
        patient_series[metric][patient] = group_sorted[metric].values
    
    # Observation group (can also be added to groupby if needed)
    if patient not in patient_groups:
        group_val = patient_program[patient_program['id пациента'].astype(str) == patient]['группа наблюдения'].values
        patient_groups[patient] = group_val[0] if len(group_val) > 0 else 'unknown'

print(f"Dictionaries created for {len(patient_dates)} patients")

# Normalization
normalized_series = {'САД': {}, 'ДАД': {}, 'ЧП': {}, 'пульсовое_давление': {}}

for metric in ['САД', 'ДАД', 'ЧП', 'пульсовое_давление']:
    for patient in all_needed_patients:
        if patient in patient_series[metric]:
            original = patient_series[metric][patient]
            if len(original) > 1 and np.std(original) > 0:
                normalized = (original - np.mean(original)) / np.std(original)
            else:
                normalized = original - np.mean(original)
            normalized_series[metric][patient] = normalized

### 1.5 Checking Normalization

In [ ]:
for metric in ['САД', 'ДАД', 'ЧП', 'пульсовое_давление']:
    if clustering_patients and metric in normalized_series:
        first_patient = clustering_patients[0]
        if first_patient in normalized_series[metric]:
            orig = patient_series[metric][first_patient]
            norm = normalized_series[metric][first_patient]
            print(f"\n{metric}:")
            print(f"  Original: mean={np.mean(orig):.2f}, std={np.std(orig):.2f}")
            print(f"  Normalized: mean={np.mean(norm):.2f}, std={np.std(norm):.2f}")

### 1.6 Creating Episodes for State Classification (by CSE)

In [ ]:
episodes = []
if len(classification_patients) > 0:
    print(f"Creating episodes for {len(classification_patients)} patients with CSE")
    
    # For progress tracking
    total_patients = len(classification_patients)
    
    for idx, patient in enumerate(classification_patients):
        # Show progress every 500 patients
        if (idx + 1) % 500 == 0:
            print(f"  Processed {idx + 1}/{total_patients} patients")
            
        # Get all CSEs for this patient
        patient_kzs = kzs[kzs['id пациента'] == patient].sort_values('дата, время формирования КЗС')
        
        if len(patient_kzs) == 0:
            continue
            
        # Get patient's time series
        patient_daily = daily_stats[daily_stats['id пациента'].astype(str) == patient].sort_values('дата')
        
        if len(patient_daily) < 14:
            continue
        
        for _, kzs_row in patient_kzs.iterrows():
            kzs_date = pd.to_datetime(kzs_row['дата, время формирования КЗС']).date()
            
            # Window 7 days BEFORE CSE
            before_data = patient_daily[
                (patient_daily['дата'].dt.date >= kzs_date - timedelta(days=7)) &
                (patient_daily['дата'].dt.date < kzs_date)
            ]
            
            # Window 7 days AFTER CSE
            after_data = patient_daily[
                (patient_daily['дата'].dt.date > kzs_date) &
                (patient_daily['дата'].dt.date <= kzs_date + timedelta(days=7))
            ]
            
            if len(before_data) >= 3:
                episodes.append({
                    'patient': patient,
                    'type': 'before_kzs',
                    'kzs_date': kzs_date,
                    'n_days': len(before_data),
                    'sad_mean': before_data['САД'].mean(),
                    'sad_std': before_data['САД'].std(),
                    'sad_min': before_data['САД'].min(),
                    'sad_max': before_data['САД'].max(),
                })
            
            if len(after_data) >= 3:
                episodes.append({
                    'patient': patient,
                    'type': 'after_kzs',
                    'kzs_date': kzs_date,
                    'n_days': len(after_data),
                    'sad_mean': after_data['САД'].mean(),
                    'sad_std': after_data['САД'].std(),
                    'sad_min': after_data['САД'].min(),
                    'sad_max': after_data['САД'].max(),
                })
    
    # Add control episodes
    np.random.seed(42)
    control_count = 0
    max_control = min(40000, len(classification_patients) // 4)  # more control episodes
    for patient in classification_patients[:max_control * 2]:  # take with reserve
        if control_count >= max_control:
            break
            
        patient_daily = daily_stats[daily_stats['id пациента'].astype(str) == patient].sort_values('дата')
        
        if len(patient_daily) < 21:
            continue
        
        patient_kzs_dates = set(pd.to_datetime(
            kzs[kzs['id пациента'] == patient]['дата, время формирования КЗС']
        ).dt.date)
        
        # Try to find up to 3 control windows per patient
        for _ in range(3):
            if control_count >= max_control:
                break
                
            max_start = len(patient_daily) - 7
            if max_start < 1:
                continue
                
            start_idx = np.random.randint(0, max_start)
            window_data = patient_daily.iloc[start_idx:start_idx+7]
            window_dates = set(window_data['дата'].dt.date)
            
            if not window_dates & patient_kzs_dates:
                episodes.append({
                    'patient': patient,
                    'type': 'control',
                    'kzs_date': None,
                    'n_days': len(window_data),
                    'sad_mean': window_data['САД'].mean(),
                    'sad_std': window_data['САД'].std(),
                    'sad_min': window_data['САД'].min(),
                    'sad_max': window_data['САД'].max(),
                })
                control_count += 1
    
    if episodes:
        episodes_df = pd.DataFrame(episodes)
        print(f"\nEpisodes created: {len(episodes_df)}")
        print(f"Distribution by type:")
        print(episodes_df['type'].value_counts())
        episodes_df.to_csv('data/ts_episodes.csv', index=False)
        print(f"Saved: data/ts_episodes.csv")
    else:
        print("Failed to create episodes")

### 1.7 Saving Data for Subsequent Stages

In [ ]:
# Form data separately for each stage
clustering_series = {
    metric: {p: normalized_series[metric][p] 
             for p in clustering_patients if p in normalized_series[metric]}
    for metric in ['САД', 'ДАД', 'ЧП', 'пульсовое_давление']
}

clustering_dates = {p: patient_dates[p] for p in clustering_patients if p in patient_dates}

ts_data = {
    # For clustering (stage 2)
    'clustering_patients': clustering_patients,
    'clustering_series': clustering_series,
    'clustering_dates': clustering_dates,
    'clustering_kzs_count': {p: kzs_count_per_patient.get(p, 0) for p in clustering_patients},
    
    # For state classification (stage 4)
    'classification_patients': classification_patients,
    'all_series': normalized_series,
    'all_dates': patient_dates,
    
    # Common
    'patient_groups': patient_groups,
    'kzs_count_per_patient': kzs_count_per_patient,
    'daily_stats': daily_stats
}

with open('data/ts_prepared_data.pkl', 'wb') as f:
    pickle.dump(ts_data, f)

print(f"\nSaved: data/ts_prepared_data.pkl")
print(f"\nContents of saved data:")
print(f"  - For clustering: {len(clustering_patients)} patients")
print(f"  - For classification: {len(classification_patients)} patients")
print(f"  - Total unique: {len(all_needed_patients)} patients")

### 1.8 Visualization of Examples (clustering set only)

In [ ]:
# Show examples from the clustering set
n_examples = min(4, len(clustering_patients))
example_patients = clustering_patients[:n_examples]

if example_patients:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, patient in enumerate(example_patients):
        ax = axes[idx]
        
        if patient in normalized_series['САД'] and patient in normalized_series['ДАД']:
            days = np.arange(len(normalized_series['САД'][patient]))
            
            ax.plot(days, normalized_series['САД'][patient], 'b-', alpha=0.7, label='SBP', linewidth=1)
            ax.plot(days, normalized_series['ДАД'][patient], 'g-', alpha=0.7, label='DBP', linewidth=1)
            
            # Add CSE information
            kzs_count = kzs_count_per_patient.get(patient, 0)
            ax.set_title(f'Patient {patient[:8]}... (CSE: {kzs_count})')
            ax.set_xlabel('Observation days')
            ax.set_ylabel('Normalized value')
            ax.legend(loc='upper right', fontsize=8)
            ax.grid(True, alpha=0.3)
            ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('data/photo/ts_clustering_examples.png', dpi=100)
    plt.show()
    print(f"Saved: data/photo/ts_clustering_examples.png")
else:
    print("No patients for visualization")

## **Patient 1 (top left):**
- **SBP**: Quite stable, small fluctuations around 0
- **DBP**: Also stable, slightly below SBP
- **CSE: 4 events** - no sharp changes visible on the graph
- **Character**: Moderate variability, without extreme peaks

## **Patient 2 (top right):**
- **SBP**: More volatile, noticeable fluctuations
- **DBP**: Follows SBP but with smaller amplitude
- **CSE: 7 events** - highest among these graphs
- **Character**: Increased variability, possible connection with frequent CSEs

## **Patient 3 (bottom left):**
- **SBP**: The most volatile! Sharp fluctuations
- **DBP**: Also unstable
- **CSE: 0 events** - interesting! High variability WITHOUT CSE
- **Character**: "Jagged" rhythm, frequent trend changes

## **Patient 4 (bottom right):**
- **SBP**: Relatively stable, smooth changes
- **DBP**: Very stable, almost a straight line
- **CSE: 0 events**
- **Character**: Calm course, minimal fluctuations

## **Key Observations:**

1. **SBP-DBP relationship** - synchrony is clearly visible everywhere (as physiologically expected)

2. **Different patterns even with the same number of CSEs**:
   - Patient 3 (0 CSE) - very unstable
   - Patient 4 (0 CSE) - very stable
   
3. **Normalization worked** - all series centered around 0

4. **Different series lengths** - from ~200 to ~400 days

## **Conclusion for clustering:**
These 4 patients will likely fall into **different clusters** because they have fundamentally different behavioral patterns. This is a good sign - it means clustering makes sense!

### 1.9 Final Report

In [ ]:
print(f"""
DATA PREPARED:

1. For CLUSTERING (stage 2 - GT-DTW):
   - Patients: {len(clustering_patients)}
   - Criterion: ≥{min_days_clustering} days of observation
   - Has CSE: yes
   - Average series length: {np.mean([len(normalized_series['САД'][p]) for p in clustering_patients if p in normalized_series['САД']]):.0f} days

2. For STATE CLASSIFICATION (stage 4):
   - Patients: {len(classification_patients)}
   - Criterion: ≥{min_days_classification} days of observation
   - Episodes created: {len(episodes_df) if 'episodes_df' in locals() else 0}

3. METRICS:
   - SBP, DBP, HR, pulse pressure
   - Normalization: Z-score (for each series separately)

FILES SAVED:
- data/ts_prepared_data.pkl - main data for stages 2 and 4
- data/ts_episodes.csv - episodes for state classification
- data/photo/ts_clustering_examples.png - examples of series

NEXT STAGE:
Implementation of GT-DTW and distance matrix calculation for {len(clustering_patients)} patients
""")

# Memory cleanup
del daily_stats, patient_program, kzs
del df
if 'episodes_df' in locals():
    del episodes_df
gc.collect()
del all_needed_patients
del clustering_series
del clustering_dates
del ts_data
plt.close('all')
del fig, axes